![image info](https://user-images.githubusercontent.com/91945811/146929832-e78ff1f4-2739-41b1-acd3-019a39b6a42c.png)

<div style="float: right; font-size: small;">
    Erstellt von Fakher Ahmed für Data Science Institut by Fabian Rappert
</div>

# Tutorial: Umgang mit fehlenden Werten in Pandas

In diesem Notebook zeigen wir, wie man einen Datensatz mit verschiedenen Missing-Value-Szenarien lädt, analysiert und gezielt bereinigt. Wir behandeln folgende Aspekte:

1. Einlesen des Datensatzes  
2. Überblick über den Datensatz und die fehlenden Werte  
3. Verschiedene Strategien zum Umgang mit fehlenden Werten:
   - Komplette Spalten löschen
   - Zeilen löschen
   - Einfache Imputation (Mean, Median, Mode)
   - Kategoriale Imputation (Mode, "Unknown")
4. Zusammenfassung


In [1]:
import pandas as pd

# Datensatz laden
df = pd.read_csv("sales_data/dataset_with_missing_values.csv")  # Pfad anpassen, falls nötig

# Erste Sicht auf den Datensatz
df.head()

,ID,Name,Gender,Department,Age,Salary,Bonus,Location
0,1,Olga,F,HR,58.0,51777.0,NaN,Berlin
1,2,Yvonne,F,Marketing,33.0,NaN,NaN,NaN
2,3,Klara,F,Finance,NaN,53483.0,NaN,NaN
3,4,Farid,M,Marketing,54.0,44541.0,3152.0,NaN
4,5,Tarek,M,Finance,59.0,53939.0,19457.0,NaN


In [2]:
display(df)

,ID,Name,Gender,Department,Age,Salary,Bonus,Location
0,1,Olga,F,HR,58.0,51777.0,NaN,Berlin
1,2,Yvonne,F,Marketing,33.0,NaN,NaN,NaN
2,3,Klara,F,Finance,NaN,53483.0,NaN,NaN
3,4,Farid,M,Marketing,54.0,44541.0,3152.0,NaN
4,5,Tarek,M,Finance,59.0,53939.0,19457.0,NaN
5,6,Ronja,F,Finance,23.0,70757.0,NaN,Hamburg
6,7,Petra,F,Finance,53.0,67065.0,995.0,Leipzig
7,8,Jonas,M,IT,34.0,59127.0,NaN,NaN
8,9,Bernd,M,HR,59.0,54276.0,NaN,Frankfurt
9,10,Walter,M,Marketing,NaN,51243.0,NaN,Köln


## Erster Überblick

Zunächst sehen wir uns den Datensatz etwas genauer an:
- **Anzahl Zeilen und Spalten**: Wie groß ist der Datensatz?
- **Datentypen**: Welche Spalten sind numerisch, welche sind kategorisch?
- **Fehlende Werte**: In welchen Spalten treten sie auf und in welchem Umfang?


In [3]:
# Übersicht zu den Spalten, Datentypen und Nicht-Null-Werten
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          20 non-null     int64  
 1   Name        20 non-null     object 
 2   Gender      18 non-null     object 
 3   Department  20 non-null     object 
 4   Age         18 non-null     float64
 5   Salary      17 non-null     float64
 6   Bonus       6 non-null      float64
 7   Location    12 non-null     object 
dtypes: float64(3), int64(1), object(4)
memory usage: 1.4+ KB


In [4]:
# Anzahl fehlender Werte pro Spalte
missing_per_column = df.isna().sum()
print("\nAnzahl fehlender Werte pro Spalte:\n", missing_per_column)


Anzahl fehlender Werte pro Spalte:
 ID             0
Name           0
Gender         2
Department     0
Age            2
Salary         3
Bonus         14
Location       8
dtype: int64


In [5]:
# Prozentualer Anteil fehlender Werte (optional)
missing_percent = df.isna().mean() * 100
print("\nProzentualer Anteil fehlender Werte:\n", missing_percent)


Prozentualer Anteil fehlender Werte:
 ID             0.0
Name           0.0
Gender        10.0
Department     0.0
Age           10.0
Salary        15.0
Bonus         70.0
Location      40.0
dtype: float64


In [6]:
# Ggf. beschreibende Statistik (numerische Spalten)
df.describe()

,ID,Age,Salary,Bonus
count,20.00000,18.000000,17.000000,6.000000
mean,10.50000,44.222222,55325.705882,13345.833333
std,5.91608,14.371359,13042.347736,8777.962939
min,1.00000,20.000000,30206.000000,995.000000
25%,5.75000,31.500000,47412.000000,6831.000000
50%,10.50000,49.500000,53939.000000,18491.500000
75%,15.25000,57.500000,67065.000000,19371.500000
max,20.00000,60.000000,78033.000000,19488.000000


## Analyse der fehlenden Werte

- **Spalte mit sehr hohem Anteil fehlender Werte** (z. B. `Bonus` ~70%):  
  → Kandidat, um die Spalte komplett zu entfernen oder sich auf spezielle Imputation zu konzentrieren.

- **Spalte mit mittlerem Anteil fehlender Werte** (z. B. `Location` ~40%):  
  → Eventuell fortgeschrittene Imputation oder "Unknown"-Kategorie.

- **Spalte mit geringem Anteil fehlender Werte** (z. B. `Salary` ~15%, `Gender` ~10%):  
  → Infrage kommen:
    - Zeilenweise löschen (falls ausreichend Daten vorhanden)  
    - Einfache Imputation (z. B. Mittelwert, Median, häufigste Kategorie)  

- **Nur sehr wenige Lücken** (z. B. `Age` mit 2 fehlenden Werten):  
  → Kaum Datenverlust beim Droppen, aber auch einfach zu imputieren.


In [7]:
# Beispiel: 'Bonus' hat ~70% Missing Values.
df_drop_bonus = df.drop(columns=["Bonus"])

In [8]:
print("Spalten im Original:", df.columns.tolist())

Spalten im Original: ['ID', 'Name', 'Gender', 'Department', 'Age', 'Salary', 'Bonus', 'Location']


In [9]:
print("Spalten nach Drop Bonus:", df_drop_bonus.columns.tolist())

Spalten nach Drop Bonus: ['ID', 'Name', 'Gender', 'Department', 'Age', 'Salary', 'Location']


In [10]:
# Prüfung, ob das sinnvolle Auswirkungen hat
df_drop_bonus.isna().sum()

ID            0
Name          0
Gender        2
Department    0
Age           2
Salary        3
Location      8
dtype: int64

**Begründung**: Wenn eine Spalte (z. B. `Bonus`) zu vielen fehlende Werte hat (z. B. 70% oder mehr), liefert sie potentiell wenig zuverlässige Information. Das Löschen kann sinnvoll sein, wenn der Informationsverlust dadurch relativ gering ist. Alternativ könnte man überlegen, ob man auf *aufwändige Imputationsmethoden* zurückgreift, falls die Spalte für das Projekt sehr wichtig ist.


In [11]:
# Beispiel: Nur 15% der Werte in 'Salary' fehlen - wir wollen Zeilen mit Missing Salary löschen
# Kopie anlegen für Vergleich
df_drop_salary_rows = df.dropna(subset=["Salary"])

In [12]:
print("Anzahl Zeilen vorher:", len(df))

Anzahl Zeilen vorher: 20


In [13]:
print("Anzahl Zeilen nach Drop:", len(df_drop_salary_rows))

Anzahl Zeilen nach Drop: 17


In [14]:
df_drop_salary_rows.isna().sum()

ID             0
Name           0
Gender         2
Department     0
Age            2
Salary         0
Bonus         11
Location       6
dtype: int64

**Begründung**: Wenn nur wenige Werte in einer Spalte fehlen, kann das zeilenweise Löschen sinnvoll sein. Man verliert wenig Daten und umgeht die potenziellen Verzerrungen oder Unsicherheiten durch Imputation. Allerdings sollte man immer bedenken, dass jede gelöschte Zeile wertvolle Informationen in den übrigen Spalten enthält – daher ist diese Methode nur bei *kleinem* Anteil an Missing Values und *ausreichender* Datenmenge empfehlenswert.




In [15]:
# Beispiel: Imputieren von 'Age' (nur 2 fehlende Werte)
df_impute_mean_age = df.copy()

In [16]:
mean_age = df_impute_mean_age["Age"].mean()
df_impute_mean_age["Age"] = df_impute_mean_age["Age"].fillna(mean_age)

# Prüfung
print("Mittelwert Alter:", mean_age)
df_impute_mean_age.isna().sum()

Mittelwert Alter: 44.22222222222222


ID             0
Name           0
Gender         2
Department     0
Age            0
Salary         3
Bonus         14
Location       8
dtype: int64

In [17]:
# Beispiel: Statt Mean kann man für 'Salary' (15% Lücken) den Median nehmen
df_impute_median_salary = df.copy()

median_salary = df_impute_median_salary["Salary"].median()
df_impute_median_salary["Salary"] = df_impute_median_salary["Salary"].fillna(median_salary)

print("Median Salary:", median_salary)
df_impute_median_salary.isna().sum()

Median Salary: 53939.0


ID             0
Name           0
Gender         2
Department     0
Age            2
Salary         0
Bonus         14
Location       8
dtype: int64

**Begründung**:  
- *Mean* (arithmetischer Mittelwert) ist die häufigste einfache Methode, doch sie kann die Verteilung verzerren, wenn Ausreißer vorhanden sind.  
- *Median* ist robuster gegenüber Ausreißern und oft sinnvoller bei schiefen Verteilungen.  
- Bei einem *geringen Prozentsatz* an fehlenden Werten oder *gleichmäßig verteilten* Missing Values ist eine einfache statistische Imputation oft unproblematisch.

Allerdings verliert man mögliche Zusammenhänge (z. B. korreliert `Salary` womöglich mit `Age` oder `Department`?), sodass fortgeschrittene Methoden genauer sein könnten.


In [18]:
# Beispiel: 'Gender' hat ca. 10% fehlende Werte.
df_impute_mode_gender = df.copy()

mode_gender = df_impute_mode_gender["Gender"].mode()[0]  # häufigste Ausprägung
df_impute_mode_gender["Gender"] = df_impute_mode_gender["Gender"].fillna(mode_gender)

df_impute_mode_gender.isna().sum()

display(df_impute_mode_gender)

,ID,Name,Gender,Department,Age,Salary,Bonus,Location
0,1,Olga,F,HR,58.0,51777.0,NaN,Berlin
1,2,Yvonne,F,Marketing,33.0,NaN,NaN,NaN
2,3,Klara,F,Finance,NaN,53483.0,NaN,NaN
3,4,Farid,M,Marketing,54.0,44541.0,3152.0,NaN
4,5,Tarek,M,Finance,59.0,53939.0,19457.0,NaN
5,6,Ronja,F,Finance,23.0,70757.0,NaN,Hamburg
6,7,Petra,F,Finance,53.0,67065.0,995.0,Leipzig
7,8,Jonas,M,IT,34.0,59127.0,NaN,NaN
8,9,Bernd,M,HR,59.0,54276.0,NaN,Frankfurt
9,10,Walter,M,Marketing,NaN,51243.0,NaN,Köln


In [ ]:
# Alternativ: Eigene Kategorie "Unknown" hinzufügen
df_unknown_gender = df.copy()
df_unknown_gender["Gender"] = df_unknown_gender["Gender"].fillna("Unknown")
display()

**Begründung** (kategoriale Spalten):
- *Mode-Imputation* ist eine einfache und oft praktikable Methode, wenn relativ wenige Werte fehlen.  
- *"Unknown"-Kategorie* kann sinnvoll sein, um fehlende Werte nicht künstlich einer vorhandenen Gruppe zuzuordnen. So bleiben Unklarheiten explizit im Datensatz.  
- Bei größeren Anteilen an fehlenden kategorischen Daten oder komplexen Zusammenhängen kann man auch *modellbasierte* Ansätze (z. B. Decision Trees) nutzen.
- In unserem Fall ist ein manuelles Ergänzen anhand der Vornamen sinvoll. Urs und Hannes isnd männliche Vornamen.

### Speicheroption des bereinigten Datensatzes

In [20]:
# 1) Bonus-Spalte droppen und Kopie erstellen
df_cleaned = df.drop(columns="Bonus").copy()

In [21]:
# 2) Fehlende Werte füllen
df_cleaned.fillna(
    {
        "Salary": df_cleaned["Salary"].median(),
        "Gender": "Unknown",
        "Age": df_cleaned["Age"].mean(),
        "Location": "Unknown",
    },
    inplace=True,
)

# Oder:

# Zeilen löschen, bspw.:
# df_cleaned.dropna(subset=['Salary'], inplace=True)

In [22]:
# 3) Kontrolle, ob alle Lücken gefüllt wurden
df_cleaned.isna().sum()

ID            0
Name          0
Gender        0
Department    0
Age           0
Salary        0
Location      0
dtype: int64

In [23]:
df.describe()

,ID,Age,Salary,Bonus
count,20.00000,18.000000,17.000000,6.000000
mean,10.50000,44.222222,55325.705882,13345.833333
std,5.91608,14.371359,13042.347736,8777.962939
min,1.00000,20.000000,30206.000000,995.000000
25%,5.75000,31.500000,47412.000000,6831.000000
50%,10.50000,49.500000,53939.000000,18491.500000
75%,15.25000,57.500000,67065.000000,19371.500000
max,20.00000,60.000000,78033.000000,19488.000000


In [24]:
df_cleaned.describe()

,ID,Age,Salary
count,20.00000,20.000000,20.000000
mean,10.50000,44.222222,55117.700000
std,5.91608,13.593944,11979.255935
min,1.00000,20.000000,30206.000000
25%,5.75000,32.500000,50285.250000
50%,10.50000,45.111111,53939.000000
75%,15.25000,56.500000,62379.750000
max,20.00000,60.000000,78033.000000


In [25]:
display(df_cleaned)

,ID,Name,Gender,Department,Age,Salary,Location
0,1,Olga,F,HR,58.000000,51777.0,Berlin
1,2,Yvonne,F,Marketing,33.000000,53939.0,Unknown
2,3,Klara,F,Finance,44.222222,53483.0,Unknown
3,4,Farid,M,Marketing,54.000000,44541.0,Unknown
4,5,Tarek,M,Finance,59.000000,53939.0,Unknown
5,6,Ronja,F,Finance,23.000000,70757.0,Hamburg
6,7,Petra,F,Finance,53.000000,67065.0,Leipzig
7,8,Jonas,M,IT,34.000000,59127.0,Unknown
8,9,Bernd,M,HR,59.000000,54276.0,Frankfurt
9,10,Walter,M,Marketing,44.222222,51243.0,Köln


### Erklärung

- **`drop(columns='Bonus')`**: Entfernt nur die `Bonus`-Spalte und gibt das Ergebnis als **neues** DataFrame zurück.  
- **`df_cleaned.fillna(..., inplace=True)`**: Füllt alle in dem Dictionary angegebenen Spalten mit den jeweiligen Werten (Median, Mean oder fixen Strings) und verändert das DataFrame direkt.  
- **`isna().sum()`**: Zeigt dir, ob nun noch fehlende Werte übrig sind.  

So ersparst du dir mehrzeilige Aufrufe für jede Spalte und hast alles **übersichtlich** in einem Schritt.

# Zusammenfassung & Empfehlung

1. **Spalten mit sehr vielen fehlenden Werten** (z. B. `Bonus`) → ggf. **Droppen**, wenn sie nicht kritisch für Analysen sind.  
2. **Einzelne, wenige fehlende Werte in wichtigen Spalten** → **Imputieren** (Mean, Median, Mode). Geringer Informationsverlust.  
3. **Kategoriale Spalten** → Modus oder "Unknown"-Kategorie.  
4. **Fortgeschrittene Methoden** → Wenn Genauigkeit besonders wichtig ist oder viele Missing Values bestehen, die nicht zufällig verteilt sind.  

Die **beste Lösung** variiert je nach:
- Umfang und Verteilung der fehlenden Werte  
- Wichtigkeit der Spalte für Analysen/Modelle  
- Größe und Struktur des Datensatzes  
- Zeit und Ressourcen, die man in die Bereinigung stecken kann  

Nach der finalen Entscheidung für bestimmte **Imputationsschritte** (oder das Droppen von Spalten/Zeilen) empfiehlt es sich, **alle weiteren Analysen** auf dem bereinigten Datensatz durchzuführen. Dadurch vermeidet man, dass unterschiedliche Versionen des Datensatzes zu unterschiedlichen Ergebnissen führen.
